# ML Study Tracker: Supervised Learning — Practical Use Cases (v2, Beginner-Annotated)

**Supervised learning** means we learn from *labeled examples*: for every row of input data (features `X`), we already know the correct answer (target `y`). The algorithm's job is to find a pattern connecting `X` to `y` so it can predict `y` for rows it has never seen.

There are two flavors you'll see throughout this notebook:

| Flavor | The target `y` is... | Example | Typical metrics |
|---|---|---|---|
| **Regression** | A continuous number | House price, tomorrow's sales | MAE, RMSE, R² |
| **Classification** | A category / class | Churn yes/no, digit 0–9 | Accuracy, Precision, Recall, F1, ROC-AUC |

### How to use this notebook
1. Run the **Common Imports** cell first — everything else depends on it.
2. Each numbered section is **self-contained**: it loads or creates its own data, so you can run any section independently after imports.
3. Read the concept markdown *before* the code, run the code, then read the **"Reading the output"** notes *after* — the numbers only teach you something if you know what to look for.
4. Tick the checklist boxes and write your own words in the Notes cells. Rewriting an idea in your own words is where the learning actually happens.

### Three ideas that appear in EVERY section (learn these first)

**1. Train/test split.** We hide a slice of the data (the *test set*) from the model during training. Performance on data the model has memorized is meaningless — a student who has seen the exam answers isn't demonstrating understanding. The test set is the "unseen exam."

**2. Data leakage.** Any situation where information from the test set (or from the future, in time series) sneaks into training. Leakage produces beautiful scores in the notebook and disasters in production. Watch for it constantly — several sections below demonstrate the *safe* pattern explicitly.

**3. Pipelines.** `Pipeline([('scaler', ...), ('model', ...)])` chains preprocessing and model into one object. The critical benefit: when you cross-validate a Pipeline, the scaler is re-fit on each training fold only — the test fold never influences the preprocessing. Fitting a scaler on ALL the data before splitting is one of the most common leakage bugs in real code.

## Table of Contents
- [Common Imports & Configuration](#Common-Imports-&-Configuration)
- [1. Linear Regression — House Price Prediction](#1.-Linear-Regression-—-House-Price-Prediction)
- [2. Logistic Regression — Customer Churn Prediction](#2.-Logistic-Regression-—-Customer-Churn-Prediction)
- [3. K-Nearest Neighbors — Scaling Sensitivity](#3.-K-Nearest-Neighbors-—-Scaling-Sensitivity)
- [4. Decision Tree — Interpretable Diagnosis Model](#4.-Decision-Tree-—-Interpretable-Diagnosis-Model)
- [5. Random Forest — Credit Risk Scoring](#5.-Random-Forest-—-Credit-Risk-Scoring)
- [6. Gradient Boosting — Credit Risk (RF Comparison)](#6.-Gradient-Boosting-—-Credit-Risk-(RF-Comparison))
- [7. SVM + PCA — High-Dimensional Digits](#7.-SVM-+-PCA-—-High-Dimensional-Digits)
- [8. Multinomial Naive Bayes — Text Classification](#8.-Multinomial-Naive-Bayes-—-Text-Classification)
- [9. Time Series Forecasting — Supervised Framing](#9.-Time-Series-Forecasting-—-Supervised-Framing)
- [10. Cross-Validation & Hyperparameter Tuning](#10.-Cross-Validation-&-Hyperparameter-Tuning)
- [Mini Project Tracker](#Mini-Project-Tracker)
- [Experiment Log](#Experiment-Log)

In [ ]:
# ============================================================
# Common Imports & Configuration
# Run this cell FIRST. Every section below assumes these exist.
# ============================================================
import numpy as np                     # numerical arrays and math
import pandas as pd                    # tabular data (DataFrames)
import matplotlib.pyplot as plt        # plotting

# Model-selection utilities:
#   train_test_split  -> carve out an unseen test set
#   cross_val_score   -> K-fold cross-validation scoring
#   GridSearchCV      -> exhaustive hyperparameter search with CV
#   TimeSeriesSplit   -> CV that respects chronological order
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, TimeSeriesSplit

# Preprocessing:
#   StandardScaler -> rescale each feature to mean 0, std 1
#   OneHotEncoder  -> turn categories ('One year') into 0/1 columns
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ColumnTransformer -> apply DIFFERENT preprocessing to different columns
# Pipeline          -> chain preprocessing + model into one leak-safe object
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Metrics — regression (errors in target units) and classification (label quality)
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve,
    confusion_matrix, classification_report
)

RANDOM_SEED = 42
# WHY a seed? ML uses randomness (splits, synthetic data, bootstraps).
# Fixing the seed makes results reproducible: same code -> same numbers.
#
# WHY no global np.random.seed() here? A single global seed makes each cell's
# output depend on the ORDER cells were run in. Instead, every section builds
# its own local generator:  rng = np.random.default_rng(RANDOM_SEED)
# so re-running any one cell always gives identical results.

## 1. Linear Regression — House Price Prediction
**Dataset Target:** California Housing Dataset (Popular Public Benchmark)

### The concept, in plain words
Linear regression assumes the target is (approximately) a *weighted sum* of the features:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n$$

Training = finding the weights $w$ that minimize squared prediction error on the training data. It's the "straight line of best fit," generalized to many dimensions.

**Why start here?** It's fast, interpretable (each weight says "one unit more of this feature adds $w$ to the prediction"), and it's the baseline every fancier model must beat.

### Three variants compared below
- **OLS (Ordinary Least Squares)** — plain linear regression, no constraints on weights.
- **Ridge** — adds a penalty on *large weights* (L2 penalty). When features are correlated, OLS weights can explode in opposite directions; Ridge shrinks them toward zero, trading a little bias for a lot of stability.
- **Lasso** — penalizes the *absolute size* of weights (L1). Its special power: it can push weights to *exactly zero*, effectively deleting features — automatic feature selection.

The `alpha` parameter controls penalty strength: `alpha=0` ≈ OLS, huge `alpha` shrinks everything to zero.

### Study Checklist
- [ ] Data Exploration & Target Skew Check — *skewed targets make squared-error models chase the long tail*
- [ ] Feature Scaling — *penalties treat all weights equally, so features must be on comparable scales*
- [ ] Model Training (OLS vs. Regularized Ridge/Lasso)
- [ ] Evaluation Metrics (RMSE, MAE, R²)
- [ ] Residual Analysis — *residual = actual − predicted; patterns in residuals reveal what the model missed*

### Metric cheat-sheet (regression)
- **MAE** — average absolute error, in target units. Easy to explain to anyone.
- **RMSE** — like MAE but squares errors first, so big misses hurt disproportionately. Also in target units.
- **R²** — fraction of target variance explained. 1.0 = perfect, 0.0 = no better than predicting the mean, negative = worse than the mean.

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: House Prices — OLS vs Ridge vs Lasso
# ============================================================
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.datasets import fetch_california_housing

# --- Load a real benchmark dataset (downloads once, then cached) ---
# 20,640 California districts. Target = median house value in units of $100k.
housing = fetch_california_housing(as_frame=True)   # as_frame=True -> pandas DataFrame
df = housing.frame

X = df.drop(columns="MedHouseVal")   # features: income, house age, rooms, location...
y = df["MedHouseVal"]                # target we want to predict

# --- Step 1: Look at the target BEFORE modeling ---
# Skewness ~0 = symmetric bell shape. Positive = long right tail.
# Squared-error models get dragged around by long tails, and this dataset
# has a known quirk: values were CAPPED at 5.0 ($500k) at collection time,
# so there's an artificial spike at the maximum.
print(f"Target skewness: {y.skew():.3f}  (rule of thumb: |skew| > 1 = strongly skewed)")
print(f"Capped at 5.0? Max = {y.max():.2f}, rows sitting at the cap = {(y == y.max()).sum()}\n")

# --- Step 2: Split BEFORE any fitting (the unseen 'exam') ---
# test_size=0.2 -> hold out 20%. random_state fixes WHICH rows are held out.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# --- Step 3: Fit three model variants under IDENTICAL preprocessing ---
# The scaler lives INSIDE the pipeline, so it is fit on training data only.
# Scaling matters here because Ridge/Lasso penalize weight sizes — a feature
# measured in tens of thousands would get an unfairly tiny weight otherwise.
candidates = {
    'OLS':   LinearRegression(),
    'Ridge': Ridge(alpha=1.0),    # alpha = penalty strength
    'Lasso': Lasso(alpha=0.01),   # smaller alpha for Lasso (L1 bites harder)
}

results = {}
for name, model in candidates.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_train, y_train)                 # learn weights on train only
    y_pred = pipe.predict(X_test)              # predict on unseen test rows
    results[name] = {
        'MAE':  mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R^2':  r2_score(y_test, y_pred),
    }
    if name == 'Ridge':
        best_pipe, best_pred = pipe, y_pred    # keep Ridge for residual analysis

print(pd.DataFrame(results).T.round(4))

# --- Step 4: Residual analysis — the model's error report card ---
# residual = actual - predicted. A healthy linear model leaves residuals that
# look like random noise: centered on 0, same spread everywhere (homoscedastic).
# A funnel/curve shape = systematic pattern the model failed to capture.
residuals = y_test - best_pred
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(best_pred, residuals, s=4, alpha=0.3)
ax[0].axhline(0, color='red', lw=1)
ax[0].set(title='Residuals vs Fitted (Ridge)', xlabel='Predicted value', ylabel='Residual')
ax[1].hist(residuals, bins=50)
ax[1].set(title='Residual Distribution', xlabel='Residual')
plt.tight_layout(); plt.show()
print(f"Mean of residuals (want ~0): {residuals.mean():.4f}")

### Reading the output (Section 1)
- The three models score **almost identically** here. That's a lesson, not a failure: with 20k rows and only 8 features, there's little overfitting for regularization to fix. Regularization shines when features are many, correlated, or data is scarce — exactly the rolling-window, small-sample settings you'll meet later.
- In the residuals-vs-fitted plot, look for the **diagonal streak of points** — that's the capped-at-5.0 rows: the model predicts a range of values but the recorded truth is stuck at 5.0. Real datasets have artifacts; residual plots find them.
- R² ≈ 0.60 means ~60% of price variance is explained by these features. Whether that's "good" depends entirely on the use case — there is no universal threshold.

## 2. Logistic Regression — Customer Churn Prediction
**Dataset Target:** Imbalanced Customer Behavior/Churn Analysis

### The concept, in plain words
Despite the name, logistic regression is a **classification** algorithm. It computes the same weighted sum as linear regression, then squashes it through the **sigmoid** function to get a probability between 0 and 1:

$$P(\text{churn}) = \frac{1}{1 + e^{-(w_0 + w_1x_1 + \dots)}}$$

You then choose a **threshold** (default 0.5) to convert probability → yes/no decision. The probability and the threshold are *separate decisions* — that separation is the whole point of this section.

### Why "imbalanced" changes everything
If only 8% of customers churn, a useless model that predicts "no churn" for everyone scores **92% accuracy**. So for imbalanced problems:
- **Precision** = of those we flagged as churners, how many actually churned? (cost of false alarms)
- **Recall** = of the actual churners, how many did we catch? (cost of missed churners)
- **F1** = harmonic mean of the two — punishes you if either is bad.
- **ROC-AUC** = threshold-free ranking quality: probability that a random churner gets a higher score than a random non-churner. 0.5 = coin flip, 1.0 = perfect ranking.

`class_weight='balanced'` tells the model: *errors on the rare class count more* — a simple, effective first response to imbalance (the alternative is resampling, e.g. SMOTE).

### The interpretability bonus: odds ratios
Each coefficient $w_i$, exponentiated ($e^{w_i}$), is an **odds ratio**: how much the *odds* of churn multiply when that feature increases by one unit (one standard deviation, after scaling). Odds ratio 1.4 = 40% higher odds; 0.5 = odds cut in half. This is why logistic regression survives in industries that demand explainable decisions.

### Study Checklist
- [ ] Imbalance Check & Evaluation Metric Selection (Avoid raw Accuracy)
- [ ] Dummy Encoding / One-Hot Encoding for Categories
- [ ] Addressing Imbalance (Class weights vs. Resampling)
- [ ] Extracting Decision Probabilities & Threshold Tuning
- [ ] Coefficient Odds-Ratio Interpretation

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: Customer Churn Prediction
# ============================================================
from sklearn.linear_model import LogisticRegression

# Local random generator -> this cell gives identical output every run,
# no matter what other cells you ran before it.
rng = np.random.default_rng(RANDOM_SEED)

# --- Step 1: Build a synthetic churn dataset with a KNOWN ground truth ---
# Synthetic data is a teaching superpower: because WE wrote the churn formula,
# we can check whether the model recovers the true drivers.
n_samples = 2000
raw_data = pd.DataFrame({
    'Age':             rng.integers(18, 70, n_samples),
    'Tenure_Months':   rng.integers(0, 72, n_samples),
    'Contract_Type':   rng.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
    'Monthly_Charges': rng.uniform(20, 120, n_samples),
})

# The TRUE churn process (the model never sees this formula, only its outcomes):
#   - base rate is low (intercept -3.2 -> rare event)
#   - month-to-month contracts churn much more (+1.8 on the log-odds scale)
#   - higher charges nudge churn up; longer tenure pulls it down
churn_logit = (-3.2
               + (raw_data['Contract_Type'] == 'Month-to-month') * 1.8
               + raw_data['Monthly_Charges'] * 0.01
               - raw_data['Tenure_Months'] * 0.03)
churn_prob = 1 / (1 + np.exp(-churn_logit))                   # sigmoid -> probability
raw_data['Churn'] = (rng.random(n_samples) < churn_prob).astype(int)  # coin flip per row

# --- Step 2: ALWAYS check imbalance before choosing metrics ---
print(f"Churn rate: {raw_data['Churn'].mean():.1%}")
print("-> Accuracy is misleading here (predicting 'nobody churns' would already")
print("   score ~92%). We will judge the model on F1 and ROC-AUC instead.\n")

# --- Step 3: Split feature types — numbers and categories need different prep ---
numeric_features     = ['Age', 'Tenure_Months', 'Monthly_Charges']
categorical_features = ['Contract_Type']
X = raw_data.drop(columns='Churn')
y = raw_data['Churn']

# stratify=y -> keep the SAME churn percentage in train and test.
# Without it, a random split could put too few churners in the test set.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

# --- Step 4: Preprocessing per column type, inside the pipeline ---
# StandardScaler for numbers; OneHotEncoder turns 'Contract_Type' into 0/1 columns.
# drop='first' drops one dummy column: with all three kept, they always sum to 1,
# which creates perfect multicollinearity and unstable coefficients.
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first'), categorical_features),
])

churn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    # class_weight='balanced': mistakes on the rare churn class are up-weighted
    # so the model can't win by ignoring churners.
    ('model', LogisticRegression(class_weight='balanced', random_state=RANDOM_SEED)),
])
churn_pipeline.fit(X_train, y_train)

# predict_proba gives [P(no churn), P(churn)] per row; [:, 1] takes P(churn).
y_pred_proba = churn_pipeline.predict_proba(X_test)[:, 1]

# --- Step 5: Threshold tuning — the probability is not the decision ---
# precision_recall_curve computes precision & recall at EVERY possible threshold.
# We compute F1 at each and pick the threshold that maximizes it.
prec, rec, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
best_t = thresholds[np.argmax(f1s)]

print(f"F1 at default threshold 0.5: {f1_score(y_test, (y_pred_proba >= 0.5).astype(int)):.4f}")
print(f"F1 at tuned threshold {best_t:.3f}: {f1s.max():.4f}")
print(f"ROC-AUC (threshold-free ranking quality): {roc_auc_score(y_test, y_pred_proba):.4f}\n")

y_pred = (y_pred_proba >= best_t).astype(int)
print(f"Precision: {precision_score(y_test, y_pred):.4f} | Recall: {recall_score(y_test, y_pred):.4f}")
# Confusion matrix layout:  [[TN, FP],
#                            [FN, TP]]
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# --- Step 6: Odds ratios — did the model recover the true story? ---
feature_names = churn_pipeline.named_steps['preprocessor'].get_feature_names_out()
coefs = churn_pipeline.named_steps['model'].coef_[0]
odds = pd.DataFrame({'coef': coefs, 'odds_ratio': np.exp(coefs)}, index=feature_names)
print("\nOdds ratios (>1 raises churn odds, <1 lowers them):")
print(odds.round(3).sort_values('odds_ratio', ascending=False))

### Reading the output (Section 2)
- **Threshold tuning matters a lot here.** With `class_weight='balanced'`, the model's probabilities are deliberately inflated for the rare class, so the F1-optimal threshold lands far from 0.5. Lesson: never assume 0.5 is right — sweep it.
- **The odds ratios recover the true story we planted**: both fixed-contract dummies show odds ratios far below 1 (relative to the dropped month-to-month baseline, they slash churn odds), tenure protects (<1), charges hurt (>1), and Age — which we gave zero true effect — sits near 1.0. When a model's coefficients match known ground truth, you can trust the machinery.
- Precision and recall at the tuned threshold are both modest — with an 8% base rate and noisy labels, that's realistic. Business would now ask: which error is more expensive, a wasted retention offer (FP) or a lost customer (FN)? That answer, not F1, picks the final threshold.

## 3. K-Nearest Neighbors — Scaling Sensitivity
**Dataset Target:** Wine Dataset (Mixed-Scale Tabular Features)

### The concept, in plain words
KNN has no training phase worth the name — it just **memorizes the training data**. To classify a new point, it finds the *k* closest training points (usually by straight-line Euclidean distance) and takes a majority vote. "You are the average of your k nearest neighbors."

### Why this section exists: the scaling trap
Euclidean distance adds up squared differences **per feature**. If one feature ranges 0–1500 (wine's `proline`) and another 0–1 (`hue`), the big feature completely dominates the distance — the model effectively ignores everything else. **Standardizing** (mean 0, std 1) puts every feature on an equal footing. The code below shows the accuracy collapse when you skip this. Any distance-based method (KNN, K-Means, SVM) has this dependency; tree-based methods do not.

### Choosing k = choosing bias vs. variance
- **k=1**: each prediction copies its single nearest neighbor — extremely flexible, extremely noise-sensitive (high variance).
- **Large k**: predictions average over many neighbors — smooth and stable, but can blur real class boundaries (high bias).
We pick k by cross-validation, not by gut feeling.

### Study Checklist
- [ ] Distance Metric Intuition (Euclidean dominance by large-scale features)
- [ ] Demonstrating Accuracy Collapse WITHOUT Scaling
- [ ] Choosing k (Bias-Variance Sweep)
- [ ] Weighted vs. Uniform Voting
- [ ] Curse of Dimensionality Awareness — *in very high dimensions, all points become nearly equidistant and "nearest" loses meaning*

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: KNN — Why Scaling Is Non-Negotiable
# ============================================================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_wine

# Wine dataset: 178 wines, 13 chemical measurements, 3 grape cultivars (classes).
# Perfect for this demo because feature scales vary WILDLY:
# 'hue' ~ 0.5-1.7 while 'proline' ~ 280-1680.
wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_SEED
)

# --- Demo 1: the same model, with and without scaling ---
knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)  # raw features
knn_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(n_neighbors=5)),
]).fit(X_train, y_train)

print(f"KNN accuracy WITHOUT scaling: {knn_raw.score(X_test, y_test):.4f}")
print(f"KNN accuracy WITH scaling:    {knn_scaled.score(X_test, y_test):.4f}")
print(f"(feature std-devs span {X.std().min():.2f} to {X.std().max():.2f} — "
      f"a ~2500x spread; unscaled distance is basically just 'proline')\n")

# --- Demo 2: sweep k with 5-fold cross-validation ---
# cross_val_score splits the TRAINING data into 5 folds, trains on 4,
# scores on the 5th, rotates, and returns 5 scores. Mean = expected accuracy,
# std = how much it wobbles across folds (stability).
for k in [1, 3, 5, 11, 21]:
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('model', KNeighborsClassifier(n_neighbors=k))])
    cv = cross_val_score(pipe, X_train, y_train, cv=5)
    print(f"k={k:>2}: CV accuracy = {cv.mean():.4f} (+/- {cv.std():.4f})")

### Reading the output (Section 3)
- Scaling lifts accuracy by ~15 percentage points **with zero change to the model**. On mixed-scale data, preprocessing IS the model improvement.
- In the k-sweep, notice the **± std** column as much as the mean: k=1 typically wobbles more across folds (variance), larger k is steadier. On this small, clean dataset large k works well; on noisier data with intricate class boundaries, k that's too large starts underfitting.
- Try it yourself: change `KNeighborsClassifier(n_neighbors=k)` to add `weights='distance'` (closer neighbors get bigger votes) and re-run the sweep — does it help small k?

## 4. Decision Tree — Interpretable Diagnosis Model
**Dataset Target:** Breast Cancer Wisconsin (Binary Diagnosis)

### The concept, in plain words
A decision tree is a flowchart of yes/no questions learned from data: *"Is worst radius ≤ 16.8? → if yes, is worst concave points ≤ 0.14? → ..."*. Each question splits the data into purer and purer groups until leaves are (mostly) one class.

**How does it pick questions?** At each node, it tries every feature and every split point, and greedily picks the split that most reduces impurity — measured by **Gini impurity** or **entropy** (both quantify "how mixed are the classes in this group"; in practice they give near-identical trees).

### The two headline properties
1. **Interpretability.** You can print the tree and read the rules — hand them to a doctor. Almost no other model offers this.
2. **Overfitting by default.** Left unbounded, a tree keeps splitting until every training point is perfectly classified — it memorizes the noise. The train/test accuracy **gap** in the sweep below is the overfitting signature. Controls: `max_depth`, `min_samples_leaf`, `ccp_alpha` (pruning).

**Bonus:** trees split on thresholds ("is x ≤ 16.8?"), so feature scaling is irrelevant to them — a welcome contrast with Section 3.

### Study Checklist
- [ ] Gini vs. Entropy Splitting Criteria
- [ ] Overfitting via Unbounded Depth (Train vs. Test Gap)
- [ ] Pruning Controls (max_depth, min_samples_leaf, ccp_alpha)
- [ ] Reading the Tree — Extracting Human Rules
- [ ] No Scaling Needed (Split Points Are Scale-Invariant)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: Decision Tree — Interpretability & Overfitting
# ============================================================
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.datasets import load_breast_cancer

# 569 tumor samples, 30 measurements each, target: malignant(0) / benign(1).
cancer = load_breast_cancer(as_frame=True)
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_SEED
)

# --- Demo 1: depth sweep — watch overfitting develop ---
# train_acc keeps climbing with depth (the tree can always memorize more),
# but test_acc peaks and then stalls/drops. The GAP between them is the
# amount of memorized noise. max_depth=None means "grow until perfect".
print("depth | train_acc | test_acc | gap (train - test)")
for depth in [2, 3, 5, 10, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_SEED).fit(X_train, y_train)
    tr, te = tree.score(X_train, y_train), tree.score(X_test, y_test)
    print(f"{str(depth):>5} |  {tr:.4f}   |  {te:.4f}  | {tr - te:+.4f}")

# --- Demo 2: print the actual learned rules ---
# THIS is why trees survive in medicine, credit, and law: the model is
# literally a checklist a human can audit. export_text prints it.
tree3 = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED).fit(X_train, y_train)
print("\nLearned rule set (top 2 levels of a depth-3 tree):")
print(export_text(tree3, feature_names=list(X.columns), max_depth=2))

### Reading the output (Section 4)
- Depth 10 and `None` hit **1.0000 train accuracy** — perfect memorization — while test accuracy is *lower* than the depth-3 tree. More capacity, worse generalization: that's overfitting in one table.
- The printed rules make clinical sense: large `worst radius` and high `worst concave points` push toward malignant. When a model's logic matches domain knowledge, that's evidence it learned signal, not noise.
- A single tree is a high-variance learner — change a few training rows and the whole flowchart can reshuffle. That fragility is exactly what the next two sections (ensembles) exist to fix.

## 5. Random Forest — Credit Risk Scoring
**Dataset Target:** Tabular Non-linear Dataset (Credit/Risk Default Profiling)

### The concept, in plain words
One deep tree overfits. A **Random Forest** trains *many* deep-ish trees and lets them vote. The magic is in making the trees *disagree* in useful ways — averaging many diverse, individually-noisy opinions cancels the noise (the "wisdom of crowds" applied to models):
1. **Bagging (bootstrap aggregating):** each tree trains on a random sample of rows, drawn *with replacement*.
2. **Feature randomness:** at every split, each tree may only consider a random subset of features (`max_features`), forcing different trees to discover different patterns.

### Two ways to ask "which features matter?"
- **MDI / Gini importance** (`feature_importances_`): how much each feature reduced impurity, summed over all splits. Free to compute, but measured on *training* data and biased toward features with many possible split points.
- **Permutation importance**: shuffle one feature's column in the *test* set and measure how much the score drops. If shuffling a feature doesn't hurt, the model wasn't really using it. Slower, but honest.
Comparing both — as we do below — is the professional habit; where they disagree is where you learn something.

### Study Checklist
- [ ] Bagging & Bootstrap Sampling Properties
- [ ] Feature Randomness Tuning (`max_features` limits)
- [ ] Tree Ensemble Depth and Node Splitting Criteria
- [ ] Evaluation Metrics via Classification Report
- [ ] MDI Gini vs. Permutation Feature Importance (both, side-by-side)

### Notes
Trees are scale-invariant, so **no StandardScaler** in this pipeline — a deliberate contrast with KNN/SVM. Putting one in wouldn't break anything, but it signals a misunderstanding of how trees work.

In [ ]:
# ============================================================
# Practical Implementation: Random Forest + Two Kinds of Importance
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.datasets import make_classification

# --- Step 1: synthesize an imbalanced 'credit default' dataset ---
# make_classification builds a labeled dataset with known structure:
#   n_informative=6 -> only 6 features truly drive the label
#   n_redundant=4   -> 4 features are just linear combos of the informative ones
#   weights=[0.85, 0.15] -> 15% positive class (defaults), like real credit data
X_raw, y_raw = make_classification(
    n_samples=1200, n_features=10, n_informative=6, n_redundant=4,
    weights=[0.85, 0.15], random_state=RANDOM_SEED
)
feature_names = [f"Feature_{i}" for i in range(10)]
df_rf = pd.DataFrame(X_raw, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    df_rf, y_raw, test_size=0.25, stratify=y_raw, random_state=RANDOM_SEED
)

# --- Step 2: fit the forest (note: NO scaler — trees don't need one) ---
rf = RandomForestClassifier(
    n_estimators=150,          # number of trees; more = smoother, slower
    max_depth=6,               # cap each tree's depth to limit memorization
    class_weight='balanced',   # up-weight the rare default class
    random_state=RANDOM_SEED,
).fit(X_train, y_train)

# classification_report prints precision/recall/F1 PER CLASS —
# always read the rare class's row, not just overall accuracy.
print("--- Random Forest Credit Risk Evaluation ---")
print(classification_report(y_test, rf.predict(X_test)))

# --- Step 3: MDI vs permutation importance, side by side ---
# permutation_importance: shuffle each feature n_repeats times on the TEST
# set and record the average score drop. Near-zero (or negative) = the model
# gets nothing real from that feature.
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_SEED)
imp = pd.DataFrame({
    'MDI_Gini':         rf.feature_importances_,   # train-side, split-based
    'Permutation_Test': perm.importances_mean,     # test-side, outcome-based
}, index=feature_names).sort_values('Permutation_Test', ascending=False)
print("Feature importance — two lenses (disagreements are instructive):")
print(imp.round(4))

### Reading the output (Section 5)
- In the classification report, the default class (`1`) has visibly lower recall than the majority class — even a balanced forest finds rare events hard. This is the number a risk manager cares about.
- **The two importance columns disagree**, and that's the lesson: MDI spreads credit across many features (redundant copies of informative ones still get used for splits), while permutation importance concentrates on the few features whose *unique* information the model actually needs — shuffling a redundant feature barely hurts because its twin is still intact.
- Features with permutation importance ≈ 0 or negative are ones the model could lose without noticing — candidates for removal in a leaner production model.

## 6. Gradient Boosting — Credit Risk (RF Comparison)
**Dataset Target:** Same Synthetic Credit Data (Direct Ensemble Comparison)

### The concept, in plain words
Random Forest builds trees **in parallel and independently**, then averages. Gradient Boosting builds trees **sequentially**: each new tree is trained to predict the *errors (residuals) of the ensemble so far*. Predictions accumulate: 

$$F_{m}(x) = F_{m-1}(x) + \eta \cdot \text{tree}_m(x)$$

where $\eta$ is the **learning rate** — how big a step each corrective tree takes.

### The knobs that matter
- **learning_rate ↔ n_estimators trade-off:** small steps need more trees. Small `learning_rate` + many trees usually generalizes best (each tree corrects gently instead of over-committing).
- **Shallow trees (depth 2–4) as weak learners:** boosting wants many *weak* correctors, not a few strong memorizers — the opposite intuition from Random Forest's deeper trees.
- **Early stopping:** hold out a validation slice (`validation_fraction`); when its loss stops improving for `n_iter_no_change` rounds, stop adding trees. This picks the effective number of trees *automatically* and prevents the sequential process from eventually fitting noise.

(XGBoost/LightGBM — which you use in your own research pipelines — are industrial-strength implementations of this same idea.)

### Study Checklist
- [ ] Boosting vs. Bagging (Sequential Error-Correction vs. Parallel Averaging)
- [ ] Learning Rate ↔ n_estimators Trade-off
- [ ] Early Stopping via Validation Fraction
- [ ] Shallow Trees as Weak Learners (depth 2–4)
- [ ] Head-to-Head Metric Comparison vs. Random Forest

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: Gradient Boosting vs Random Forest
# ============================================================
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.datasets import make_classification

# Same synthetic credit data as Section 5 (regenerated here so this cell
# is self-contained and runnable on its own).
X_raw, y_raw = make_classification(
    n_samples=1200, n_features=10, n_informative=6, n_redundant=4,
    weights=[0.85, 0.15], random_state=RANDOM_SEED
)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.25, stratify=y_raw, random_state=RANDOM_SEED
)

models = {
    # Bagging ensemble: 150 independent, moderately deep trees, averaged.
    'RandomForest': RandomForestClassifier(
        n_estimators=150, max_depth=6, class_weight='balanced',
        random_state=RANDOM_SEED),

    # Boosting ensemble: up to 300 SHALLOW trees (depth 3), each taking a
    # small corrective step (learning_rate=0.05), with early stopping:
    # 15% of training data is held aside; if validation loss doesn't improve
    # for 20 consecutive trees, training stops.
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=3,
        validation_fraction=0.15, n_iter_no_change=20,
        random_state=RANDOM_SEED),
}

print("model            | F1     | ROC-AUC")
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]   # P(default) per test row
    print(f"{name:<16} | {f1_score(y_test, model.predict(X_test)):.4f}"
          f" | {roc_auc_score(y_test, proba):.4f}")

gb = models['GradientBoosting']
print(f"\nEarly stopping halted GB at {gb.n_estimators_} of 300 allowed trees")
print("-> the validation slice decided the model size, not us.")

### Reading the output (Section 6)
- On this dataset the two ensembles land close together — RF slightly ahead on AUC, GB on F1 (or vice versa depending on seed). The honest takeaway: **on modest tabular problems, well-tuned bagging and boosting are competitive**; boosting's edge grows with more data and careful tuning.
- Early stopping used only ~⅓ of the allowed trees. Without it, the remaining ~200 trees would slowly fit validation noise. Letting held-out data decide model size is the same discipline as early-stopping a neural network.
- Note what GB lacks here: `class_weight`. sklearn's `GradientBoostingClassifier` doesn't support it — handling imbalance requires `sample_weight` in `.fit()` or switching to XGBoost/LightGBM (`scale_pos_weight`). Library limitations are part of real model selection.

## 7. SVM + PCA — High-Dimensional Digits
**Dataset Target:** Handwritten Digits (High-Dimensional Feature Matrices)

### The concept, in plain words
A **Support Vector Machine** looks for the separating boundary with the **maximum margin** — the widest possible "street" between classes. Only the points on the street's edges (the *support vectors*) determine the boundary; everything else could vanish without changing the model.

**The kernel trick:** a linear boundary can't separate everything. Kernels let the SVM behave *as if* the data were mapped into a much higher-dimensional space — where a linear separator exists — without ever computing that mapping. The **RBF (radial basis function)** kernel effectively draws smooth, curved boundaries; `C` controls tolerance for misclassified points (small C = wide, forgiving street; large C = narrow street, fits training data harder) and `gamma` controls boundary wiggliness.

**Why PCA in front?** The digits are 8×8 images = 64 pixel features, many nearly redundant (corner pixels are almost always blank). PCA (fully explained in the Unsupervised notebook) compresses to the directions holding 95% of the variance — fewer, denoised features, faster SVM. Crucially, PCA sits **inside the Pipeline**, so it's learned from training folds only. Fitting PCA on all data first would leak test-set structure into training.

### Study Checklist
- [ ] Multi-dimensional Spatial Feature Layout
- [ ] PCA Variance Retention as Preprocessing (inside the Pipeline = no leakage)
- [ ] Maximizing Margin Planes via Support Vectors
- [ ] Evaluating RBF vs. Linear Hyperplane Kernels (head-to-head)
- [ ] Hyperparameter Sensitivity (C, gamma)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: SVM + PCA — Linear vs RBF Kernels
# ============================================================
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

# 1,797 grayscale 8x8 digit images. Each image is flattened into 64 pixel
# intensities -> a 64-dimensional feature vector per sample.
digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.2,
    stratify=digits.target,            # keep all 10 digits proportionally present
    random_state=RANDOM_SEED
)

# Compare kernels under an IDENTICAL preprocessing chain.
# Pipeline order: scale -> PCA -> SVM. Because all three live in one Pipeline,
# .fit() learns scaler AND PCA from training data only. No leakage.
for kernel in ['linear', 'rbf']:
    pipe = Pipeline([
        ('scaler', StandardScaler()),                       # SVMs are distance-based -> scale
        ('pca', PCA(n_components=0.95,                      # keep components explaining 95% var
                    random_state=RANDOM_SEED)),
        ('model', SVC(kernel=kernel, C=2.0, gamma='scale',  # gamma='scale' = sensible default
                      random_state=RANDOM_SEED)),
    ]).fit(X_train, y_train)

    acc = accuracy_score(y_test, pipe.predict(X_test))
    kept = pipe.named_steps['pca'].n_components_
    print(f"{kernel:>6} kernel: accuracy = {acc:.4f} | PCA kept {kept} of 64 components")

### Reading the output (Section 7)
- PCA kept ~40 of 64 components — a third of the dimensions were carrying almost no information (mostly always-blank border pixels).
- RBF edges out linear, but only slightly: after scaling and PCA, digits are *nearly* linearly separable. The gap between kernels is the measure of how non-linear a problem really is — on messier data it widens dramatically.
- Extension exercise: wrap this pipeline in `GridSearchCV` (Section 10) over `model__C: [0.1, 1, 10]` and `model__gamma: [0.001, 0.01, 'scale']`. The `step__param` naming convention is how you tune *inside* a pipeline.

## 8. Multinomial Naive Bayes — Text Classification
**Dataset Target:** 20 Newsgroups Subset (NLP Multi-Class Target)

### The concept, in plain words
Naive Bayes classifies with Bayes' theorem: $P(\text{class} \mid \text{words}) \propto P(\text{class}) \times P(\text{words} \mid \text{class})$. The **"naive"** part is assuming every word appears independently of every other word given the class — obviously false for language ("machine" and "learning" travel together), yet the classifier works remarkably well anyway, because it only needs the *ranking* of class probabilities to be right, not their exact values.

### From text to numbers: TF-IDF
Models need numeric input. **TF-IDF** turns each document into a vector where each word's value = (how often it appears in *this* document) × (how rare it is *across all* documents). Common glue words score near zero; distinctive words ("hockey", "orbital") score high. `max_features=1000` keeps only the 1,000 strongest words — a crude but effective dimensionality cap.

### Laplace smoothing (`alpha`) — the parameter we sweep
If the word "goalie" never appeared in any training document of class `sci.space`, its estimated probability there is zero — and one zero multiplies the *entire* class probability to zero. Smoothing adds a pseudo-count `alpha` to every word/class pair so nothing is ever exactly zero. Tiny `alpha` trusts training counts (risk: brittleness on unseen words); large `alpha` flattens all words toward equal probability (risk: ignoring the evidence).

### Study Checklist
- [ ] Prior Probabilities Allocation Models
- [ ] TF-IDF Calculations
- [ ] Conditional Feature Independence Assumptions
- [ ] Laplace Smoothing (`alpha` sweep implemented)
- [ ] Confusion Matrix Error Pattern Reading

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: Multinomial Naive Bayes with alpha Sweep
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.datasets import fetch_20newsgroups

# Three well-separated newsgroup topics (downloads once, then cached).
# remove=('headers','footers','quotes') strips email metadata — WITHOUT this,
# the model 'cheats' by memorizing sender addresses instead of learning topics.
# (A classic real-world leakage story.)
categories = ['sci.space', 'comp.graphics', 'rec.sport.hockey']
news_train = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers', 'footers', 'quotes'))
news_test  = fetch_20newsgroups(subset='test', categories=categories,
                                remove=('headers', 'footers', 'quotes'))

# --- Sweep the smoothing parameter ---
# alpha near 0: trust raw counts (would CRASH at exactly 0 on unseen words).
# alpha large:  over-smooth, evidence gets diluted.
for alpha in [0.01, 0.1, 0.5, 1.0, 5.0]:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=1000,        # keep 1000 strongest words
                                  stop_words='english')),   # drop 'the', 'and', ...
        ('model', MultinomialNB(alpha=alpha)),
    ]).fit(news_train.data, news_train.target)              # raw strings in — pipeline vectorizes
    acc = accuracy_score(news_test.target, pipe.predict(news_test.data))
    print(f"alpha={alpha:<5}: test accuracy = {acc:.4f}")

# --- Full report at a chosen alpha ---
best = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000, stop_words='english')),
    ('model', MultinomialNB(alpha=0.5)),
]).fit(news_train.data, news_train.target)
y_pred = best.predict(news_test.data)

print("\n", classification_report(news_test.target, y_pred,
                                   target_names=news_train.target_names))
# Confusion matrix: rows = true class, columns = predicted class.
# Off-diagonal cells show WHICH topics get confused with which.
print("Confusion Matrix (rows=true, cols=predicted):")
print(confusion_matrix(news_test.target, y_pred))

### Reading the output (Section 8)
- The alpha sweep is typically **flat-topped**: a broad range of moderate alphas performs similarly, with degradation at the extremes. Robustness to its main hyperparameter is one reason NB makes a great first baseline for any text task.
- In the confusion matrix, `sci.space` ↔ `comp.graphics` confuse each other more than either confuses with `hockey` — technical vocabularies overlap; sports vocabulary doesn't. Off-diagonal structure tells you *which* classes need better features, not just *how many* errors you made.
- Modern context: for production text classification you'd now reach for transformer embeddings — but NB + TF-IDF trains in milliseconds, needs no GPU, and sets the bar any expensive model must clearly beat to justify its cost.

## 9. Time Series Forecasting — Supervised Framing
**Dataset Target:** Supermarket Sales / Historical Demand Sequences

### The concept, in plain words
A time series has no ready-made `X` and `y` — just one column of ordered values. The trick that converts forecasting into supervised learning: **make the past the features**.
- `Lag_1` = yesterday's value; `Lag_7` = the value one week ago
- `Rolling_Mean_3` = average of the previous 3 days (recent level); `Rolling_Std_7` = recent volatility

Now every row is (recent history → today's value): an ordinary regression problem.

### The two iron rules
1. **No future leakage in features.** Note the pattern `shift(1).rolling(3)` — shift FIRST, then roll, so the window covers days *t−3…t−1*, never day *t* itself. `rolling(3)` without the shift would include today's value in a feature used to predict today: perfect leakage, garbage model.
2. **Chronological split, never random.** Random shuffling puts future rows in the training set — the model "trains on the future and predicts the past," inflating scores meaninglessly. Always: train on the earliest span, test on the final span.

### The rule everyone skips: beat the naive baselines
- **Persistence:** tomorrow = today (i.e., just predict `Lag_1`).
- **Seasonal naive:** tomorrow = same day last week (predict `Lag_7`).
These cost nothing and are shockingly hard to beat on many real series (stock prices most famously). A model that can't beat them has learned nothing — always report them alongside your model.

### Study Checklist
- [ ] Structural Decomposition (Trend, Seasonality, Residual Noise)
- [ ] Designing Lag Features without Future Data Leakage
- [ ] Adding Rolling Windows (Mean, Std)
- [ ] Executing Chronological Train/Test Time Split
- [ ] Beating the Naive Baselines (persistence & seasonal-naive)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: Forecasting vs Naive Baselines
# ============================================================
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(RANDOM_SEED)

# --- Step 1: build a synthetic series with KNOWN structure ---
# Real sales series are typically: trend + seasonality + noise.
# Building it ourselves means we know exactly what a good model should find.
t = np.arange(120)                                   # 120 days
trend       = 0.3 * t                                # slow upward drift
seasonality = 10 * np.sin(2 * np.pi * t / 7)         # weekly cycle (period 7)
noise       = rng.normal(0, 1.5, 120)                # irreducible randomness
s = pd.DataFrame({'Sales': 100 + trend + seasonality + noise})

# --- Step 2: leakage-safe feature engineering ---
# shift(1) moves everything down one row -> row t sees value from t-1.
# CRITICAL pattern: shift(1).rolling(k) -> window covers t-k .. t-1 ONLY.
s['Lag_1']          = s['Sales'].shift(1)                    # yesterday
s['Lag_7']          = s['Sales'].shift(7)                    # a week ago (captures seasonality)
s['Rolling_Mean_3'] = s['Sales'].shift(1).rolling(3).mean()  # recent level
s['Rolling_Std_7']  = s['Sales'].shift(1).rolling(7).std()   # recent volatility
s.dropna(inplace=True)   # first rows lack history -> NaN -> drop them

# --- Step 3: CHRONOLOGICAL split — final 14 days held out, NO shuffling ---
split_idx = len(s) - 14
feats = ['Lag_1', 'Lag_7', 'Rolling_Mean_3', 'Rolling_Std_7']
X_train, X_test = s[feats].iloc[:split_idx], s[feats].iloc[split_idx:]
y_train, y_test = s['Sales'].iloc[:split_idx], s['Sales'].iloc[split_idx:]

# --- Step 4: model vs the two mandatory baselines ---
ts_preds = LinearRegression().fit(X_train, y_train).predict(X_test)
persistence    = X_test['Lag_1'].values   # baseline 1: 'tomorrow = today'
seasonal_naive = X_test['Lag_7'].values   # baseline 2: 'tomorrow = last week'

print("model            | MAE    | RMSE")
for name, pred in [('Persistence', persistence),
                   ('Seasonal Naive', seasonal_naive),
                   ('LinearRegression', ts_preds)]:
    print(f"{name:<16} | {mean_absolute_error(y_test, pred):.4f}"
          f" | {np.sqrt(mean_squared_error(y_test, pred)):.4f}")

# --- Visualize the final 14-day forecast ---
plt.figure(figsize=(10, 3.5))
plt.plot(y_train.index[-30:], y_train.iloc[-30:], label='history (last 30d)')
plt.plot(y_test.index, y_test, label='actual', marker='o', ms=3)
plt.plot(y_test.index, ts_preds, label='model forecast', marker='x', ms=4)
plt.legend(); plt.title('14-Day Holdout Forecast'); plt.tight_layout(); plt.show()

### Reading the output (Section 9)
- The ranking tells a story: **persistence is terrible** (weekly seasonality means yesterday is a poor guide to today), **seasonal naive is decent** (it captures the cycle for free), and **the model roughly halves seasonal naive's MAE** — because it combines the seasonal signal (`Lag_7`) with the trend (`Rolling_Mean_3` rises over time).
- If your model had only beaten *persistence* but not *seasonal naive*, it would have learned less than a one-line heuristic — that's why we report both.
- Warning for real data: this synthetic series is stationary-ish and friendly. Real sales have holidays, promotions, and regime changes; real financial returns are mostly noise (expect models to barely edge out naive baselines, if at all — a lesson directly relevant to return-forecasting research).

## 10. Cross-Validation & Hyperparameter Tuning
**Dataset Target:** California Housing (Reusing Section 1 Pipeline)

### The concept, in plain words
**Hyperparameters** (Ridge's `alpha`, KNN's `k`, tree depth...) are knobs the training process cannot learn — *we* must choose them. But choosing them by test-set score is silent cheating: the test set is supposed to simulate the future, and tuning on it means the "future" quietly leaked into your decisions.

**K-fold cross-validation** solves this: split the *training* data into K folds; train on K−1, score on the held-out fold, rotate K times, average. Every training row gets used for both training and validation — no separate tuning set wasted, no test set touched.

**GridSearchCV** = try every hyperparameter combination × K folds, keep the best, then refit the winner on all training data. Because we pass it a whole **Pipeline**, the scaler is re-fit inside each fold — the leak-safe pattern from the intro, now doing real work.

### The three numbers not to confuse
- `best_params_` — the winning knob settings.
- `best_score_` — average CV score of the winner (an estimate, slightly optimistic since we picked the max).
- **Held-out test score** — the number you report. Computed exactly once, at the very end, on data that influenced nothing.

### Ordered data needs ordered CV
KFold shuffles — catastrophic for time series (trains on the future). **TimeSeriesSplit** creates expanding windows: train[0..n] → test[n+1..m], grow, repeat. This is precisely the rolling-window logic used in serious backtesting and panel research.

### Study Checklist
- [ ] K-Fold CV Mechanics (fit K times, average out-of-fold scores)
- [ ] GridSearchCV Over a Full Pipeline (scaler refit per fold = no leakage)
- [ ] Reading `best_params_` vs. `best_score_` vs. Held-Out Test Score
- [ ] TimeSeriesSplit for Ordered Data (never KFold on time series)
- [ ] Nested-CV Awareness (why test-set tuning is silent overfitting)

### Notes
Write your understanding, formulas, assumptions, mistakes, and examples here.

In [ ]:
# ============================================================
# Practical Implementation: GridSearchCV Over a Pipeline
# ============================================================
from sklearn.linear_model import Ridge
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
X = housing.frame.drop(columns="MedHouseVal")
y = housing.frame["MedHouseVal"]

# The test set is carved out FIRST and then not touched until the last line.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge())])

# Naming convention: '<step_name>__<param_name>' routes the value to the right
# pipeline step. 'model__alpha' -> the alpha of the step we named 'model'.
grid = GridSearchCV(
    pipe,
    param_grid={'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},  # 5 candidates
    cv=5,                                       # x 5 folds = 25 fits total
    scoring='neg_root_mean_squared_error',      # sklearn maximizes -> negate RMSE
    n_jobs=-1,                                  # use all CPU cores
).fit(X_train, y_train)   # CV happens INSIDE training data; test set untouched

print(f"Best alpha (best_params_):  {grid.best_params_['model__alpha']}")
print(f"Best CV RMSE (best_score_): {-grid.best_score_:.4f}   <- tuning estimate")

# The ONE final look at the test set. Report THIS number.
test_rmse = np.sqrt(mean_squared_error(y_test, grid.predict(X_test)))
print(f"Held-out test RMSE:         {test_rmse:.4f}   <- the honest number\n")

# --- Correct CV geometry for ORDERED data ---
# Each fold trains on an expanding past window, tests on the block right after.
# Compare with KFold, which would happily train on the future.
print("TimeSeriesSplit fold boundaries (illustrated on 100 ordered points):")
for i, (tr, te) in enumerate(TimeSeriesSplit(n_splits=3).split(np.arange(100))):
    print(f"  fold {i}: train[0..{tr[-1]}] -> test[{te[0]}..{te[-1]}]")

### Reading the output (Section 10)
- CV RMSE and held-out test RMSE land close together — the sign of a healthy workflow. A big gap (CV much better than test) means your tuning overfit the CV folds, usually from searching too many combinations on too little data.
- Ridge's alpha barely matters on this dataset (all candidates score similarly) — consistent with Section 1's finding that regularization has little to fix here. A flat grid is a finding, not a failure.
- The TimeSeriesSplit printout shows the expanding-window shape: `train[0..24]→test[25..49]`, `train[0..49]→test[50..74]`, ... This is the exact geometry of walk-forward backtesting — the bridge between textbook CV and real quantitative research.

## Mini Project Tracker

| Project | Topic | Dataset Benchmark | Model Baseline | Primary Metric | Status | Notes |
|---|---|---|---|---|---|---|
| **House Price Prediction** | Regression | California Housing | `Pipeline(Scaler, Ridge)` vs OLS/Lasso | RMSE / R² | Not Started | Skew check + residual homoscedasticity plot |
| **Customer Churn** | Classification (Imbalanced) | Synthetic Tabular | `Pipeline(ColumnTransformer, LogReg)` | F1 (tuned threshold) / ROC-AUC | Not Started | Odds ratios + threshold sweep |
| **KNN Scaling Demo** | Classification | Wine | `Pipeline(Scaler, KNN)` | Accuracy | Not Started | Show unscaled accuracy collapse |
| **Tree Interpretability** | Classification | Breast Cancer | `DecisionTreeClassifier(max_depth=3)` | Accuracy + Rule Extraction | Not Started | Depth sweep = overfitting demo |
| **Credit Risk (RF)** | Classification (Imbalanced) | Synthetic `make_classification` | `RandomForestClassifier` (no scaler) | F1 / Classification Report | Not Started | MDI vs permutation importance |
| **Credit Risk (GB)** | Classification | Same Synthetic | `GradientBoostingClassifier` + early stop | F1 / ROC-AUC | Not Started | Head-to-head vs RF |
| **Digits SVM** | Classification | load_digits | `Pipeline(Scaler, PCA, SVC)` | Accuracy | Not Started | Linear vs RBF kernel |
| **News Text NB** | NLP Classification | 20 Newsgroups | `Pipeline(TfidfVectorizer, MultinomialNB)` | Accuracy / Confusion Matrix | Not Started | Alpha smoothing sweep |
| **Sales Forecasting** | Time Series | Synthetic Sequence | `LinearRegression(Lags + Rolling)` | MAE vs Naive Baselines | Not Started | Must beat persistence & seasonal naive |
| **Tuning Workflow** | Meta / Validation | California Housing | `GridSearchCV(Pipeline)` | CV RMSE vs Test RMSE | Not Started | TimeSeriesSplit awareness |

## Experiment Log

| Date | Problem Context | Dataset Input | Model Configuration | Preprocessing Setup | Metric Tracked | Result | Next Progressive Steps |
|---|---|---|---|---|---|---|---|
| YYYY-MM-DD | House Price Prediction | California Housing | Ridge (alpha=1.0) | Standard Scaled | RMSE / R² | | Compare vs GridSearchCV-tuned alpha |
| YYYY-MM-DD | Customer Churn | Synthetic (8% positive) | LogReg (balanced) | ColumnTransformer | F1 @ tuned threshold | | Try SMOTE resampling vs class weights |